In [ ]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent / "src"))
from paths import RAW, PROCESSED, RESULTS, savefig

Renewable Potentials: Weather (ERA5 + atlite)

Uses a pre-downloaded, already-prepared ERA5 `atlite.Cutout` for Austria (`data/raw/AU-2019.nc`)
and, combined with the solar/wind availability masks from `02_land_eligibility_solar.ipynb` and
`03_land_eligibility_wind.ipynb`, computes wind and solar capacity factor time series per model
region.

No CDS API key is needed here since the cutout was already downloaded and prepared externally —
loading it is just `atlite.Cutout(path=...)`, no `cutout.prepare()` call.

In [ ]:
!pip install atlite

In [ ]:
import atlite
import geopandas as gpd
import xarray as xr
from atlite.gis import ExclusionContainer

Load Model Regions

In [ ]:
# Regions produced by 01_regions_centroids.ipynb
gadm = gpd.read_file(PROCESSED / "gadm_aut_level1.geojson")
regions = gadm.set_index("NAME_1")

Load ERA5 Weather Data from the Pre-Downloaded Cutout

In [ ]:
cutout_path = RAW / "AU-2019.nc"

cutout = atlite.Cutout(path=str(cutout_path))
cutout

Rebuild Solar and Wind Availability (same exclusions as notebooks 02 and 03)

In [ ]:
landcover_path = (
    RAW
    / "copernicus-glc"
    / "PROBAV_LC100_global_v3.0.1_2019-nrt_Discrete-Classification-map_EPSG-4326-AT.tif"
)
suitable_land_codes = [20, 30, 40, 60]
wdpa_path = RAW / "wdpa" / "WDPA_Oct2022_Public_shp-AUT.tif"
elevation_path = RAW / "gebco" / "GEBCO_2014_2D-AT.nc"

regions_m = regions.to_crs("EPSG:3035")

# Solar excluder (protected areas + suitable land cover)
solar_excluder = ExclusionContainer(crs=3035, res=100)
solar_excluder.add_raster(wdpa_path, nodata=255)
solar_excluder.add_raster(landcover_path, codes=suitable_land_codes, invert=True, nodata=255)

# Wind excluder (protected areas + suitable land cover + airports + roads + built-up + elevation)
wind_excluder = ExclusionContainer(crs=3035, res=100)
wind_excluder.add_raster(wdpa_path, nodata=255)
wind_excluder.add_raster(landcover_path, codes=suitable_land_codes, invert=True, nodata=255)

airports = gpd.read_file(RAW / "ne_10m_airports.gpkg").to_crs(regions_m.crs)
airports_aut = gpd.sjoin(airports, regions_m[["geometry"]], how="inner", predicate="within")
wind_excluder.add_geometry(airports_aut.geometry, buffer=10_000)

roads = gpd.read_file(RAW / "ne_10m_roads.gpkg").to_crs(regions_m.crs)
roads_aut = gpd.sjoin(roads, regions_m[["geometry"]], how="inner", predicate="intersects")
wind_excluder.add_geometry(roads_aut.geometry, buffer=300)

wind_excluder.add_raster(landcover_path, codes=[50], buffer=1000, nodata=255)
wind_excluder.add_raster(elevation_path, codes=lambda x: x > 2000, crs=4326)

Availability Matrix per Region (on the ERA5 cutout grid)

In [ ]:
A_solar = cutout.availabilitymatrix(regions, solar_excluder)
A_wind = cutout.availabilitymatrix(regions, wind_excluder)

Deployment Density -> Capacity Matrix (3 MW/km2 for both wind and solar)

In [ ]:
CAP_PER_SQKM = 3  # MW/km^2

cell_area = cutout.grid.set_index(["y", "x"]).to_crs(3035).area / 1e6  # km^2
cell_area = xr.DataArray(cell_area, dims=("spatial",))

solar_capacity_matrix = A_solar.stack(spatial=["y", "x"]) * cell_area * CAP_PER_SQKM
wind_capacity_matrix = A_wind.stack(spatial=["y", "x"]) * cell_area * CAP_PER_SQKM

In [ ]:
# Total installable capacity per region (MW) - needed to normalize power output back to a capacity factor
solar_capacity_mw = solar_capacity_matrix.sum("spatial")
wind_capacity_mw = wind_capacity_matrix.sum("spatial")

Onshore Wind Capacity Factor Time Series

(Austria is landlocked, so no offshore wind region applies here — the
`NREL_ReferenceTurbine_5MW_offshore` reference turbine from the assignment is not used.)

In [ ]:
wind_generation = cutout.wind(
    matrix=wind_capacity_matrix,
    turbine="Vestas_V112_3MW",
    index=regions.index,
)  # MW per region

# atlite's `matrix` weighting returns power output (MW), not a capacity factor -
# divide by the region's installable capacity to normalize back to 0-1
wind_cf = wind_generation / wind_capacity_mw

wind_cf.to_netcdf(PROCESSED / "wind_capacity_factors_AUT.nc")

Solar PV Capacity Factor Time Series

In [ ]:
solar_generation = cutout.pv(
    panel="CdTe",
    orientation="latitude_optimal",
    matrix=solar_capacity_matrix,
    index=regions.index,
)  # MW per region

# same normalization as wind: power output (MW) -> capacity factor (0-1)
solar_cf = solar_generation / solar_capacity_mw

solar_cf.to_netcdf(PROCESSED / "solar_capacity_factors_AUT.nc")

Plots

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(12, 5))
wind_cf.to_pandas().plot(ax=ax)
ax.set_title("Onshore Wind Capacity Factor Time Series per Region")
ax.set_ylabel("Capacity factor")

savefig(fig, "weather", "wind_capacity_factors.png")
plt.show()

In [ ]:
wind_cf_df = wind_cf.to_pandas()
region_names = list(regions.index)

ncols = 3
nrows = -(-len(region_names) // ncols)  # ceil division

fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 3 * nrows), sharex=True, sharey=True)
axes = axes.flatten()

for ax, region in zip(axes, region_names):
    wind_cf_df[region].plot(ax=ax, color="tab:blue")
    ax.set_title(region)
    ax.set_ylabel("Capacity factor")
    ax.set_ylim(0, 1)

for ax in axes[len(region_names):]:
    ax.axis("off")

fig.suptitle("Onshore Wind Capacity Factor Time Series by Region")
fig.tight_layout()

savefig(fig, "weather", "wind_capacity_factors_by_region.png")
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
solar_cf.to_pandas().plot(ax=ax)
ax.set_title("Solar PV Capacity Factor Time Series per Region")
ax.set_ylabel("Capacity factor")

savefig(fig, "weather", "solar_capacity_factors.png")
plt.show()

In [ ]:
solar_cf_df = solar_cf.to_pandas()

fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 3 * nrows), sharex=True, sharey=True)
axes = axes.flatten()

for ax, region in zip(axes, region_names):
    solar_cf_df[region].plot(ax=ax, color="tab:orange")
    ax.set_title(region)
    ax.set_ylabel("Capacity factor")
    ax.set_ylim(0, 1)

for ax in axes[len(region_names):]:
    ax.axis("off")

fig.suptitle("Solar PV Capacity Factor Time Series by Region")
fig.tight_layout()

savefig(fig, "weather", "solar_capacity_factors_by_region.png")
plt.show()